# 🔀 Hybrid Retrieval

**Day 8 — Notebook 1 of 4 | Estimated Time: 35 minutes**

---

## What You'll Learn
- Why dense retrieval alone sometimes fails
- BM25: the classic keyword-based ranking algorithm
- Combining BM25 + dense retrieval with Reciprocal Rank Fusion (RRF)
- When hybrid retrieval outperforms either method alone

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb rank-bm25

In [ ]:
import sys, os
import numpy as np
import chromadb
from rank_bm25 import BM25Okapi

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Why Dense Retrieval Sometimes Fails

Dense (semantic) retrieval is great at understanding meaning — but it has blind spots:

| Query type | Dense retrieval | BM25 keyword |
|-----------|----------------|---------------|
| "How does photosynthesis work?" | ✅ Understands meaning | ✅ Matches keywords |
| "What is HNSW?" (acronym) | ⚠️ May not map acronym correctly | ✅ Finds exact string |
| "Python 3.12 release date" (specific version) | ⚠️ May match wrong Python docs | ✅ Finds exact version |
| "Error: NullPointerException at line 47" | ⚠️ Generic similarity may miss | ✅ Exact error string |
| Conceptual question about a domain | ✅ Handles paraphrase well | ⚠️ May miss if keywords differ |

**Hybrid retrieval** fuses both signals, getting the best of both worlds.

---

## 2. BM25 — The Keyword Ranking Algorithm

BM25 (Best Match 25) is the standard for keyword-based retrieval, used by Elasticsearch, Solr, and Lucene.

It scores documents based on:
- **Term frequency (TF)** — how often the query term appears in the document
- **Inverse document frequency (IDF)** — how rare the term is across all documents
- **Document length normalisation** — penalises very long documents for having many matches

```
BM25(query, doc) = Σ IDF(term) × (TF(term, doc) × (k1 + 1)) / (TF(term, doc) + k1 × (1 - b + b × |doc| / avgdl))
```

In [ ]:
# Our test corpus — a mix of technical docs
CORPUS = [
    {"id": "0",  "text": "Python 3.12 was released in October 2023. Key features include improved error messages, a new type parameter syntax, and performance improvements of up to 5% over Python 3.11."},
    {"id": "1",  "text": "Python 3.11 introduced specialised adaptive interpreter and was released in October 2022. It offered 10-60% performance improvements over Python 3.10."},
    {"id": "2",  "text": "HNSW (Hierarchical Navigable Small World) is a graph-based approximate nearest neighbour algorithm used in vector databases like ChromaDB and Pinecone."},
    {"id": "3",  "text": "A NullPointerException in Java occurs when code attempts to use a null reference where an object is required. It is one of the most common runtime errors."},
    {"id": "4",  "text": "Photosynthesis is the biological process by which green plants convert carbon dioxide and water into glucose and oxygen using sunlight."},
    {"id": "5",  "text": "Machine learning models learn patterns from labelled training data. The quality and quantity of training data is the most important factor in model performance."},
    {"id": "6",  "text": "The Transformer architecture uses self-attention mechanisms to process sequences. It was introduced in 'Attention is All You Need' (Vaswani et al., 2017)."},
    {"id": "7",  "text": "BM25 stands for Best Match 25 and is a probabilistic ranking function used in information retrieval. It is the default ranking algorithm in Elasticsearch."},
    {"id": "8",  "text": "Docker containers package applications and their dependencies into isolated environments. They ensure consistent behaviour across development and production."},
    {"id": "9",  "text": "Retrieval Augmented Generation (RAG) combines a retrieval system with a generative language model. The retriever fetches relevant documents; the generator uses them as context."},
]

texts = [d["text"] for d in CORPUS]

# Build BM25 index
tokenized = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized)

print(f"BM25 index built over {len(texts)} documents")

In [ ]:
def bm25_search(query: str, top_k: int = 5) -> list[dict]:
    """BM25 keyword search."""
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [
        {**CORPUS[i], "score": float(scores[i]), "rank": r + 1}
        for r, i in enumerate(ranked)
    ]


# Test BM25
for q in ["Python 3.12 release", "NullPointerException", "What is HNSW?"]:
    results = bm25_search(q, top_k=3)
    print(f"BM25 — '{q}'")
    for r in results:
        print(f"  [{r['score']:.2f}] {r['text'][:80]}...")
    print()

---

## 3. Dense Retrieval with ChromaDB

In [ ]:
# Build dense index
chroma = chromadb.Client()
col = chroma.create_collection("hybrid_corpus", metadata={"hnsw:space": "cosine"})

result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[d["id"] for d in CORPUS],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
)
print(f"Dense index built: {col.count()} documents")


def dense_search(query: str, top_k: int = 5) -> list[dict]:
    """Dense semantic search."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values

    results = col.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents", "distances"],
    )
    return [
        {
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
            "rank": i + 1,
        }
        for i in range(len(results["ids"][0]))
    ]


# Compare on tricky queries
for q in ["Python 3.12 release", "NullPointerException", "What is HNSW?"]:
    results = dense_search(q, top_k=3)
    print(f"Dense — '{q}'")
    for r in results:
        print(f"  [{r['score']:.3f}] {r['text'][:80]}...")
    print()

---

## 4. Reciprocal Rank Fusion (RRF)

**RRF** is the standard algorithm for merging ranked lists from multiple retrieval systems. It rewards documents that appear in **high positions across multiple lists**.

```
RRF_score(doc) = Σ  1 / (k + rank_i(doc))
                 i

where k=60 is a smoothing constant, rank_i is the position in list i
```

A document ranked #1 in one list and #3 in another scores higher than a document ranked #1 in one list but absent from the other.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[dict]],
    id_key: str = "id",
    k: int = 60,
    top_k: int = 5,
) -> list[dict]:
    """
    Fuse multiple ranked lists using Reciprocal Rank Fusion.
    Each list item must have an 'id' and a 'text' field.
    """
    scores = {}  # id → RRF score
    docs = {}    # id → document text

    for ranked_list in ranked_lists:
        for rank, item in enumerate(ranked_list, start=1):
            doc_id = item[id_key]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            docs[doc_id] = item["text"]

    # Sort by RRF score descending
    sorted_ids = sorted(scores, key=scores.get, reverse=True)
    return [
        {"id": doc_id, "text": docs[doc_id], "rrf_score": scores[doc_id]}
        for doc_id in sorted_ids[:top_k]
    ]


def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    """Hybrid retrieval: BM25 + dense, fused with RRF."""
    # Get candidates from each method (retrieve more than needed for fusion)
    bm25_results = bm25_search(query, top_k=top_k * 2)
    dense_results = dense_search(query, top_k=top_k * 2)

    # BM25 results need an 'id' field — add index if missing
    for r in bm25_results:
        if "id" not in r:
            r["id"] = str(texts.index(r["text"]))

    return reciprocal_rank_fusion([bm25_results, dense_results], top_k=top_k)


print("✅ Hybrid search ready")

---

## 5. Head-to-Head Comparison

In [ ]:
test_queries = [
    {"query": "Python 3.12 release date",         "expected_id": "0"},
    {"query": "NullPointerException Java",         "expected_id": "3"},
    {"query": "What is HNSW used for?",            "expected_id": "2"},
    {"query": "How does photosynthesis work?",     "expected_id": "4"},
    {"query": "graph algorithm vector databases",  "expected_id": "2"},
    {"query": "combining retrieval with generation","expected_id": "9"},
]

print(f"{'Query':<42} {'BM25':>6} {'Dense':>7} {'Hybrid':>8}")
print("-" * 70)

bm25_hits = dense_hits = hybrid_hits = 0

for item in test_queries:
    q, expected = item["query"], item["expected_id"]

    bm25_top = bm25_search(q, top_k=3)
    dense_top = dense_search(q, top_k=3)
    hybrid_top = hybrid_search(q, top_k=3)

    b_hit = any(r["id"] == expected for r in bm25_top)
    d_hit = any(r["id"] == expected for r in dense_top)
    h_hit = any(r["id"] == expected for r in hybrid_top)

    bm25_hits += b_hit
    dense_hits += d_hit
    hybrid_hits += h_hit

    b_icon = "✅" if b_hit else "❌"
    d_icon = "✅" if d_hit else "❌"
    h_icon = "✅" if h_hit else "❌"

    print(f"{q[:41]:<42} {b_icon}      {d_icon}       {h_icon}")

n = len(test_queries)
print("-" * 70)
print(f"{'Recall@3':<42} {bm25_hits/n:.0%}     {dense_hits/n:.0%}      {hybrid_hits/n:.0%}")

---

## 6. Hybrid RAG Pipeline

In [ ]:
def hybrid_rag(question: str, top_k: int = 3) -> str:
    """Full hybrid RAG: retrieve with BM25+dense fusion, generate with Gemini."""
    chunks = hybrid_search(question, top_k=top_k)
    context = "\n\n".join(f"[Doc {c['id']}] {c['text']}" for c in chunks)

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer using only the provided context. Cite document IDs.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


questions = [
    "What improvements did Python 3.12 bring over 3.11?",
    "What algorithm does Elasticsearch use for ranking?",
    "How does RAG combine retrieval and generation?",
]

for q in questions:
    print(f"❓ {q}")
    print(f"🤖 {hybrid_rag(q)}")
    print("-" * 60)

---

## 🏋️ Exercise 1: Tune the RRF k Parameter

The `k` parameter in RRF controls how much weight is given to rank position. Experiment with different values.

In [ ]:
# Exercise 1: RRF k tuning
# TODO:
# 1. Run hybrid_search with k values: 10, 30, 60, 100
# 2. For each k, measure Recall@3 across all test_queries above
# 3. Which k gives the best recall?
# Note: Lower k = rank position matters more. Higher k = more uniform weight.

for k_val in [10, 30, 60, 100]:
    hits = 0
    for item in test_queries:
        bm25_r = bm25_search(item["query"], top_k=6)
        dense_r = dense_search(item["query"], top_k=6)
        # TODO: Call reciprocal_rank_fusion with this k_val and check if expected_id is in top-3
        pass
    # print(f"k={k_val}: Recall@3 = {hits/len(test_queries):.0%}")

---

## 🏋️ Exercise 2: Weighted Fusion

Instead of treating BM25 and dense equally, give each a weight. This lets you tune how much to trust each signal.

In [ ]:
# Exercise 2: Weighted hybrid fusion
# TODO: Implement weighted_hybrid_search(query, bm25_weight=0.3, dense_weight=0.7)
# Strategy: normalise each set of scores to [0,1], then combine as:
#   final_score = bm25_weight * bm25_norm_score + dense_weight * dense_norm_score
# Compare Recall@3 at different weight ratios: (0.1, 0.9), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3)

def weighted_hybrid_search(query: str, bm25_weight: float = 0.3, dense_weight: float = 0.7, top_k: int = 3) -> list[dict]:
    """Weighted combination of BM25 and dense scores."""
    # TODO: Implement
    pass


# for weights in [(0.1, 0.9), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]:
#     hits = sum(...)
#     print(f"BM25={weights[0]}, Dense={weights[1]}: Recall@3 = {hits/len(test_queries):.0%}")

---

## Key Takeaways

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **BM25** | Exact keyword matching, acronyms, specific strings | Misses paraphrases, no semantic understanding |
| **Dense** | Semantic similarity, paraphrase-robust | Can miss exact keywords, acronyms |
| **Hybrid (RRF)** | Best of both worlds | Slightly more complex, two indexes to maintain |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| BM25 Explained | Blog | [elastic.co/blog/practical-bm25-part-2](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables) |
| Reciprocal Rank Fusion Paper | Paper | [dl.acm.org/doi/10.1145/1571941.1572114](https://dl.acm.org/doi/10.1145/1571941.1572114) |

---

**Next up:** [02_Query_Transformation.ipynb](./02_Query_Transformation.ipynb)